# Phase 02 — Penggabungan Bank, Pemecahan Kalimat & EDA

Notebook **mandiri (self-contained)**. Mengubah data mentah artikel (hasil Phase 01) menjadi data **per kalimat** yang siap untuk BERTopic (Phase 03), sekaligus melakukan **EDA** di level artikel dan kalimat.

### Alur
1. **Baca** `data/raw/raw.csv` (hasil scrape 4 bank).
2. **Gabung keyword → kolom `bank`** tunggal (BCA/BBCA→BCA, BRI/BBRI→BRI, BNI/BBNI→BNI, Mandiri/BMRI→Mandiri). Satu artikel bisa menyebut >1 bank, dicatat dipisah koma (mis. `"BCA, Mandiri"`).
3. **Cleaning ringan** (buang 'Baca:', footer research, normalisasi spasi). Isi tetap mentah secukupnya — TANPA stemming/stopword (merusak embedding BERTopic).
4. **Pecah per kalimat** dengan pemecah yang melindungi singkatan Indonesia (`PT.`, `Tbk.`, angka desimal, dll). Simpan `id_artikel` + `id_kalimat` + `bank`.
5. **Inspeksi kalimat < 5 kata SEBELUM dibuang** — pastikan yang dibuang memang sampah, bukan kalimat relevan.
6. **Buang kalimat < 5 kata** (filter KUALITAS, bukan tematik — tidak menyaring topik agar aspek BERTopic tetap induktif).
7. **EDA** level artikel & level kalimat.
8. **Output** `data/processed/sentences.csv`.

> Prinsip: kalimat TIDAK difilter berdasarkan kata relevan. Kalimat tak relevan nanti dibuang lewat klaster outlier BERTopic (Phase 03), bukan dibuang di sini.

## 1. Import & baca data

In [ ]:
import re
from pathlib import Path
import pandas as pd

input_csv = "data/raw/raw.csv"          # <- bisa diubah: sumber data mentah (hasil Phase 01)
df = pd.read_csv(input_csv, dtype={"id_artikel": str})
print(f"Artikel mentah: {len(df)}")
df.head(3)

## 2. Gabung keyword → kolom `bank`

Keyword scraping (BCA, BBCA, BRI, ...) digabung jadi 4 bank. Deteksi dari **judul + isi** karena satu artikel sering menyebut beberapa bank. Hasil: kolom `bank` berisi daftar bank yang disebut (dipisah koma).

In [ ]:
# Pola deteksi tiap bank (nama + kode saham). \b = batas kata agar tidak salah tangkap.
POLA_BANK = {
    "BCA":     r"\bbca\b|\bbbca\b|bank central asia",          # <- bisa diubah: pola BCA
    "BRI":     r"\bbri\b|\bbbri\b|bank rakyat indonesia|brilink",
    "BNI":     r"\bbni\b|\bbbni\b|bank negara indonesia",
    "Mandiri": r"\bmandiri\b|\bbmri\b|bank mandiri",
}

def deteksi_bank(teks):
    teks = (teks or "").lower()
    return ", ".join(b for b, pat in POLA_BANK.items() if re.search(pat, teks))

df["bank"] = (df["judul"].fillna("") + " " + df["isi_mentah"].fillna("")).map(deteksi_bank)

print("Distribusi jumlah bank yang disebut per artikel:")
print(df["bank"].map(lambda x: len(x.split(", ")) if x else 0).value_counts().sort_index())
print("\nArtikel tanpa bank terdeteksi:", (df["bank"] == "").sum())
df[["id_artikel", "keyword", "bank", "judul"]].head(5)

## 3. Cleaning ringan

Buang noise struktural yang mengganggu pemecahan kalimat & BERTopic: tautan 'Baca:'/'Pilihan Redaksi', footer research, email, normalisasi spasi. **Tidak** membuang isi berdasarkan topik.

In [ ]:
def cleaning_ringan(teks):
    t = str(teks)
    t = re.sub(r"CNBC INDONESIA RESEARCH\s*\S*@\S*", " ", t, flags=re.I)   # footer research
    t = re.sub(r"\S+@\S+\.\S+", " ", t)                                    # email
    t = re.sub(r"Baca\s*:\s*", " ", t, flags=re.I)                         # label 'Baca:'
    t = re.sub(r"Pilihan Redaksi", " ", t, flags=re.I)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["isi_bersih"] = df["isi_mentah"].map(cleaning_ringan)
print("Contoh sebelum:", df["isi_mentah"].iloc[0][:150])
print("\nContoh sesudah:", df["isi_bersih"].iloc[0][:150])

## 4. Pemecah kalimat (melindungi singkatan Indonesia)

Pemecah naif memotong salah di `PT.`, `Tbk.`, angka desimal (`1.519`). Fungsi ini **melindungi** titik pada singkatan umum dulu, baru memecah di `. ! ?`.

In [ ]:
# Singkatan yang titiknya BUKAN akhir kalimat
SINGKATAN = ["PT", "Tbk", "Persero", "Rp", "US", "No", "Jl", "Dr", "Ir", "Hj",
             "dll", "dsb", "dkk", "a.l", "yakni"]   # <- bisa diubah: tambah singkatan bila perlu

def pecah_kalimat(teks):
    t = re.sub(r"\s+", " ", str(teks)).strip()
    for ab in SINGKATAN:                                   # lindungi 'PT.' -> 'PT<DOT>'
        t = re.sub(rf"\b{re.escape(ab)}\.", f"{ab}<DOT>", t)
    t = re.sub(r"(?<=\d)\.(?=\d)", "<DOT>", t)              # lindungi desimal 1.519
    kalimat = re.split(r"(?<=[.!?])\s+", t)                 # pecah di . ! ?
    return [k.replace("<DOT>", ".").strip() for k in kalimat if k.strip()]

# Pecah semua artikel -> tabel kalimat
baris = []
for _, r in df.iterrows():
    for i, k in enumerate(pecah_kalimat(r["isi_bersih"])):
        baris.append({
            "id_artikel": r["id_artikel"],
            "id_kalimat": f"{r['id_artikel']}_{i}",
            "bank": r["bank"],
            "timestamp": r["timestamp"],
            "kalimat": k,
            "n_kata": len(k.split()),
        })
kal = pd.DataFrame(baris)
print(f"Total kalimat dari {len(df)} artikel: {len(kal)} (rata {len(kal)/len(df):.1f} kalimat/artikel)")

## 5. Inspeksi kalimat < 5 kata (SEBELUM dibuang)

Pastikan yang akan dibuang memang sampah (footer, penomoran, label 'Baca'), bukan kalimat relevan.

In [ ]:
AMBANG_KATA = 5   # <- bisa diubah: ambang minimum kata per kalimat

pendek = kal[kal["n_kata"] < AMBANG_KATA]
print(f"Kalimat < {AMBANG_KATA} kata: {len(pendek)} ({len(pendek)/len(kal)*100:.1f}% dari total)")
print("Distribusi jumlah kata:", dict(pendek["n_kata"].value_counts().sort_index()))
print("\n--- 40 contoh kalimat yang akan DIBUANG ---")
for s in pendek["kalimat"].head(40).tolist():
    print("  |", s)

## 6. Buang kalimat < 5 kata

In [ ]:
kal_final = kal[kal["n_kata"] >= AMBANG_KATA].reset_index(drop=True)
print(f"Kalimat sebelum filter : {len(kal)}")
print(f"Kalimat setelah filter : {len(kal_final)} (dibuang {len(kal)-len(kal_final)})")

## 7. EDA

### 7a. Level artikel

In [ ]:
print(f"Jumlah artikel         : {len(df)}")
print(f"id_artikel unik        : {df['id_artikel'].nunique()} (duplikat: {len(df)-df['id_artikel'].nunique()})")
print(f"Panjang isi (char)     : rata {df['isi_mentah'].str.len().mean():.0f} | min {df['isi_mentah'].str.len().min()} | max {df['isi_mentah'].str.len().max()}")

# distribusi per bank (satu artikel bisa dihitung di >1 bank)
from collections import Counter
c = Counter()
for b in df["bank"]:
    for x in (b.split(", ") if b else []):
        c[x] += 1
print("\nArtikel yang menyebut tiap bank:")
for bank, n in c.most_common():
    print(f"  {bank:10} {n}")

# distribusi waktu (per bulan)
df["bulan"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.to_period("M").astype(str)
print("\nJumlah artikel per bulan (10 teratas):")
print(df["bulan"].value_counts().sort_index().tail(10))

### 7b. Level kalimat

In [ ]:
print(f"Total kalimat (final)      : {len(kal_final)}")
print(f"Rata kalimat per artikel   : {len(kal_final)/df['id_artikel'].nunique():.1f}")
print(f"Panjang kalimat (kata)     : rata {kal_final['n_kata'].mean():.1f} | min {kal_final['n_kata'].min()} | max {kal_final['n_kata'].max()}")
print("\nDistribusi panjang kalimat (kata):")
bins = [5, 10, 15, 20, 30, 50, 1000]
print(pd.cut(kal_final["n_kata"], bins=bins, right=False).value_counts().sort_index())

## 8. Simpan output

In [ ]:
output_csv = Path("data/processed/sentences.csv")   # <- bisa diubah: lokasi output
output_csv.parent.mkdir(parents=True, exist_ok=True)
kolom = ["id_kalimat", "id_artikel", "bank", "timestamp", "kalimat"]
kal_final[kolom].to_csv(output_csv, index=False)
print(f"[SIMPAN] {len(kal_final)} kalimat -> {output_csv}")
kal_final[kolom].head(10)